In [14]:
!pip install -q streamlit pyngrok google-genai

In [22]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 4.1 MB/s eta 0:00:00


In [ ]:
!pip install -q langchain-groq langgraph langchain

In [ ]:
from pyngrok import ngrok
from google.colab import userdata

ngrok.set_auth_token(userdata.get('ngrok'))
print("ngrok token berhasil dikonfigurasi!")

ngrok token berhasil dikonfigurasi!


In [108]:
import subprocess
import time

def run_streamlit(filename, port=8501):
    subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)

    subprocess.run(["fuser", "-k", f"{port}/tcp"], capture_output=True)

    ngrok.kill()

    time.sleep(3)

    proc = subprocess.Popen(
        [
            "streamlit", "run", filename,
            "--server.headless=true",
            "--server.port", str(port),
            "--server.enableCORS=false",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    time.sleep(3)

    public_url = ngrok.connect(port)
    print(f"Streamlit berjalan: {public_url}")

    return proc

In [126]:
%%writefile streamlit_app_basic.py
import streamlit as st
import os
from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver

st.title("911 customer service - AI Agent untuk menangani kasus Emergency")

st.sidebar.header("Konfigurasi API")
groq_key = st.sidebar.text_input("Masukkan Groq API Key:", type="password", placeholder="gsk_...")

if "order_db" not in globals():
    order_db = []

if not groq_key:
    st.warning("Silakan masukkan Groq API Key Anda di sidebar terlebih dahulu untuk memulai.")
else:
    system_instruction = """
    You are an emergency customer service 911 system.
    A human will talk to you about emergency situations, and you will match them with the available emergency services (Instansi).

    CRITICAL RULE FOR EMPATHY & DISTRESS PHRASES (KALIMAT PEDULI & TENANG):
    When a user expresses panic, fear, or asks for help using phrases like "tolong saya", "bantu saya", "tolong", "darurat", "mohon bantuannya", or similar distressed sentences, you MUST always start your response with a calm, empathetic, and caring tone.
    Example tone: "Saya di sini untuk membantu Anda. Apa yang sedang terjadi?" if the user has not told the incident then "Bantuan akan segera dikoordinasikan. Ceritakan apa yang terjadi."
    After showing concern, immediately identify the emergency situation and direct the person to the relevant agency.

    CRITICAL RULE FOR SIMILAR WORDS / SYNONYMS (PENTING):
    You must understand words that are similar, synonyms, or slang used by users. For example:
    - Words like "rampok", "maling", "begal", "todong", "jambret", "pencuri" mean "Pencurian" -> route to Polisi.
    - Words like "kebakaran", "api", "asap", "ledakan", "korsleting" mean "Kebakaran" -> route to Pemadam.
    - Words like "hanyut", "tenggelam", "longsor", "gempa", "tersesat" mean "Pencarian manusia/penyelamatan" -> route to SAR.
    - Words like "tabrakan", "jantungan", "pingsan", "darurat medis", "sakit berat", "melahirkan" mean "Gawat darurat/kecelakaan" -> route to Ambulans.

    CRITICAL RULE FOR ACCIDENTS (Kecelakaan):
    When a user reports an accident ("kecelakaan", "tabrakan beruntun", "luka"), you must add MULTIPLE services at the same time: specifically "Ambulans" and "Polisi" (and "Pemadam" if there is smoke/fire).

    CRITICAL RULE FOR ACCIDENTS (Manusia hilang):
    When a user reports an accident ("orang hilang", "Manusia hilang", "bunuh diri", "percobaan melukai diri"), you must add MULTIPLE services at the same time: specifically "SAR" and "Polisi" (and "Pemadam" if there is smoke/fire).

    CRITICAL RULE FOR ACCIDENTS (Serangan Hewan):
    When a user reports an accident ("digigit hewan", "diserang hewan", "terkena racun hewan"), you must add MULTIPLE services at the same time: specifically "Pemadam" and "Ambulans".

    CRITICAL RULE FOR ACCIDENTS (Kriminal):
    When a user reports an accident ("dibacok", "dipukul", "terkena miras", "penusukan", "penembakan"), you must add MULTIPLE services at the same time: specifically "Ambulans" and "Polisi".

    CRITICAL RULE FOR OFF-TOPIC:
    If the user asks about anything completely off-topic, strictly refuse and say:
    "Maaf, saya adalah customer service 911 dan hanya dapat membantu Anda seputar panggilan darurat. Ada yang ingin Anda laporkan?"

    CRITICAL RULE FOR OUT-OF-MENU ITEMS:
    If a request is completely unrelated to emergency services, politely refuse using this template:
    "Maaf, untuk saat ini kami tidak memiliki instansi [nama item]. Sebagai alternatif, kami dapat mengerahkan [sebutkan 1-2 instansi alternatif dari menu]."

    Add services to the customer's order with add_to_order, and reset the order with clear_order.
    To see the contents of the order so far, call get_order.
    Always confirm_order with the user before calling place_order.
    Once place_order has returned, check if there are multiple services or ongoing emergencies. If multiple services are requested, inform the user: "bantuan sedang dalam perjalanan ke lokasi Anda."
    """

    def get_menu() -> str:
        """Provide the latest up-to-date menu."""
        return """
      Instansi & Kategori Darurat:

      1. Polisi (Security & Penegakan Hukum):
        - Kasus: Pencurian, maling, begal, jambret, perampokan, perundungan (bullying), ancaman pembunuhan, teror, kerusuhan, tawuran, KDRT, orang hilang karena tindak kriminal.

      2. Pemadam (Fire & Rescue Kebencanaan):
        - Kasus: Kebakaran gedung/hutan, ledakan gas, korsleting listrik parah, evakuasi sarang tawon berbahaya, penyelamatan hewan terjebak (kucing di atas pohon, anjing di saluran air).

      3. SAR (Search and Rescue / Pencarian Khusus):
        - Kasus: Pencarian manusia hilang di hutan/gunung, orang hanyut di sungai, evakuasi korban bencana alam (gempa bumi, longsor, banjir besar), kecelakaan pesawat/kapal.

      4. Ambulans (Medical Emergency / Kesehatan):
        - Kasus: Gawat darurat medis mendadak, kecelakaan lalu lintas, tabrakan, korban pingsan, serangan jantung, sesak napas parah, proses melahirkan darurat, keracunan akut.
      """

    def add_to_order(item: str) -> str:
        """Add an item to the customer's order."""
        global order_db
        order_db.append(item)
        return f"I've added '{item}' to your order."

    def clear_order() -> str:
        """Clear all items from the customer's order."""
        global order_db
        order_db.clear()
        return "Your order has been cleared."

    def get_order() -> list[str]:
        """Get the current items in the customer's order."""
        global order_db
        return order_db

    def confirm_order() -> str:
        """Confirm the order with the customer."""
        global order_db
        if not order_db:
            return "Your order is currently empty. What can I get for you?"
        order_string = ", ".join(order_db)
        return f"Your order contains: {order_string}. Is this correct?"

    def place_order() -> str:
        """Place the final order."""
        global order_db
        if not order_db:
            return "There's nothing in your order to place."
        final_order_summary = ", ".join(order_db)
        order_db.clear()
        return f"Your order for '{final_order_summary}' has been placed! It will be ready shortly."

    customer_services_tools = [get_menu, add_to_order, clear_order, get_order, confirm_order, place_order]

    @st.cache_resource
    def load_agent(api_key):
        model = ChatGroq(
            temperature=0.2,
            model_name="openai/gpt-oss-20b",
            groq_api_key=api_key
        )
        checkpointer = InMemorySaver()
        return create_react_agent(model, customer_services_tools, prompt=system_instruction, checkpointer=checkpointer)

    try:
        agent = load_agent(groq_key)

        config = {"configurable": {"thread_id": "streamlit_user_session"}}

        if "messages" not in st.session_state:
            st.session_state.messages = []
            st.session_state.messages.append({"role": "assistant", "content": "Halo! Selamat datang di 911 Customer Services. Ada yang bisa saya bantu hari ini?"})

        for message in st.session_state.messages:
            with st.chat_message(message["role"]):
                st.markdown(message["content"])

        if user_input := st.chat_input("Ketik Urgensi Anda di sini..."):
            st.session_state.messages.append({"role": "user", "content": user_input})
            with st.chat_message("user"):
                st.markdown(user_input)

            with st.chat_message("assistant"):
                with st.spinner("Customer Services sedang meracik jawaban..."):
                    try:
                        response = agent.invoke(
                            {"messages": [{"role": "user", "content": user_input}]},
                            config=config
                        )
                        bot_response = response["messages"][-1].content
                        st.markdown(bot_response)
                        st.session_state.messages.append({"role": "assistant", "content": bot_response})
                    except Exception as e:
                        st.error(f"Terjadi kesalahan pada Agent/API Groq: {e}")

    except Exception as e:
        st.error(f"Gagal menginisialisasi Agent: {e}")

Overwriting streamlit_app_basic.py


Setelah file ditulis, jalankan dengan ngrok (abaikan Warning)

In [127]:
proc = run_streamlit("streamlit_app_basic.py")

PyngrokNgrokHTTPError: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: The endpoint 'https://wrought-press-luminous.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"}}


In [110]:
import os, signal
from pyngrok import ngrok
try:
    proc.terminate()
    print("App dihentikan.")

    ngrok.kill()
except:
    pass